# The three arms

**Spends money.** Set `ARMS` and `PRUNES`, read section 2, then turn `DRY_RUN`
off.

`ARMS` and `PRUNES` are lists and every pair of them runs, so the number to
approve is the total at the bottom of section 2. Three arms against four
filters is twelve sweeps, and on GDC the paged ones are most of the cost.

`rerank` is Magneto: the model sees the retriever's top `TOP_K` in one prompt
and cannot decline, so its recall is capped at the retriever's recall@`TOP_K`.

`gated` shows the same block and allows the reply NONE. `rerank` against
`gated` is what declining is worth at fixed depth.

`paged` shows `WINDOW` candidates at a time; the model commits or asks for the
next block. `gated` against `paged` is what depth is worth. Cost per column:

| the column | blocks it opens |
|---|---|
| match in the first window | 1 |
| match deep in the tail | `ceil(rank / WINDOW)` |
| no match anywhere | `ceil(n_targets / WINDOW)`, so 37 on GDC |

404 of GDC's 569 source columns have no gold target, so the last row is the
common case.

`prompt_layout` is fixed by the one-call arms (`oneshot` for rerank,
`oneshot_none` for gated); changing it makes a one-block paged run.

## 0. What to run

In [17]:
import sys
from pathlib import Path

import pandas as pd

# Wide enough that `arm` is not collapsed into an ellipsis.
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))

from corpora import load
from evaluation import scoring as report
from schemaoracle import cache, pruning
from schemaoracle.bench import Bench

# --- what to run ----------------------------------------------------------
CORPUS = "gdc"                  # or "valentine", "valentine/opendata", ...
STUDIES = None                  # None = every pair in the corpus
REPS = 1                        # runs per configuration; 3 for a spread

# Every arm runs against every filter in PRUNES, all priced together in
# section 2. Cheapest first, so an interrupted sweep has whole arms done.
ARMS = ["rerank", "gated", "paged"]

TOP_K = 20                      # the shortlist rerank and gated are shown

# --- paged only; ignored by rerank and gated ------------------------------
WINDOW = 20                     # candidates per block
# A block scoring below this is declined; 0.00 leaves it to the model.
THRESHOLDS = [0.00]

# --- the filter -----------------------------------------------------------
#   None               every column asked, every candidate offered
#   "mutual"           skip a source column with no mutual nearest neighbour
#                      (41% of the calls on GDC)
#   "zero"             drop targets no source is close to; smaller blocks
#   ("mutual","zero")  both
# Each filter is its own configuration in the filename and gets its own Bench.
# `bench.cost` quotes the unfiltered worst case.
#PRUNES = [None, "mutual", "zero", "zerosrc", ("mutual","zero"), ("zerosrc","zero")]
PRUNES = [None]

# What a walk does with an unreadable reply: "stop" leaves the column
# unanswered; "fallback" keeps the retriever's window, as rerank does. Both
# send the same requests, so this is not in the filename.
ON_UNPARSEABLE = "stop"

# Re-ask an unreadable block up to this many times, with the format spelled
# out. Tagged into the variant as `retry{n}`; seed those files first with
# `reseed.plan` + `reseed.write`, or section 1 refuses to run.
MAX_PARSE_RETRIES = 3

# `match_retriever` shows the text the order was built from; `magneto` uses
# Magneto's own. Either way it is in the variant.
TARGET_TEXT = "match_retriever"

# True prices the sweep and sends nothing.
DRY_RUN = False

# label -> the four knobs that decide the order. Labels must be unique: a
# cached order refuses to be served under a different configuration.
RETRIEVERS = {
    "qwen": dict(embedding_model="Qwen/Qwen3-Embedding-4B",
                 encoding_mode="header_values_xml",
                 sampling_mode="mixed", sampling_size=10, seed=0),
    "magneto": dict(embedding_model="vida-nyu/magneto-schema-retriever-gdc",
                     encoding_mode="header_values_default",
                     sampling_mode="mixed", sampling_size=10, seed=0),
}

# `client` goes to build_client; on Ollama, `think=False` and `num_ctx` both
# change the answer. A `jev-` model uses the decisions endpoint instead: pin
# the version, keep MAX_PARSE_RETRIES at 0 (a typed answer cannot fail to
# parse) and run paged at threshold 0.00 only, since its scores are one
# distribution per block.
MODELS = [
    #dict(provider="anthropic", model="claude-sonnet-5",
    #     prompt_layout="standard"),
    #dict(provider="openrouter", model="jev-1.13", prompt_layout="standard"),
    #dict(provider="ollama", model="deepseek-v4-flash:cloud",
    #     prompt_layout="standard", client=dict(think=False, timeout_s=600.0)),
    dict(provider="ollama", model="deepseek-v4.1-flash:cloud",
         prompt_layout="standard", client=dict(think=False, timeout_s=600.0)),
    #dict(provider="ollama", model="deepseek-v4-pro:cloud",
    #     prompt_layout="standard", client=dict(think=False, timeout_s=600.0)),
    #dict(provider="openrouter", model="anthropic/claude-sonnet-5",
    #     record_as="anthropic", prompt_layout="standard"),
]

def levels_for(arm):
    """The knob this arm sweeps. Only `paged` has a threshold to vary."""
    return THRESHOLDS if arm == "paged" else [TOP_K]


PAIRS = load(CORPUS, names=STUDIES)

# filter tag -> the Bench that writes under it.
BENCHES = {pruning.tag(prune): Bench(
    corpus=CORPUS, pairs=PAIRS,
    results=REPO / "results" / "runs",
    retrievers=RETRIEVERS, models=MODELS, target_text=TARGET_TEXT,
    reps=REPS, window=WINDOW, seed=0, on_unparseable=ON_UNPARSEABLE,
    prune=prune, max_parse_retries=MAX_PARSE_RETRIES,
    dry_run=DRY_RUN,
    resume=True, use_cache=True, verbose=True,
) for prune in PRUNES}

# (arm, filter tag, bench), arm by arm.
JOBS = [(arm, tag, BENCHES[tag]) for arm in ARMS for tag in BENCHES]

# For helpers that read only a frame, such as `health`.
BENCH = next(iter(BENCHES.values()))


def label(arm, tag):
    return f"{arm}, prune={tag or 'none'}"


BENCH.banner()
print(f"\narms       {ARMS}")
print(f"filters    {[t or 'none' for t in BENCHES]}")
print(f"\n{len(JOBS)} job(s), all of them priced together in section 2:")
for arm, tag, _ in JOBS:
    print(f"  {label(arm, tag):<28} levels={levels_for(arm)}")

env        nothing loaded from .env
corpus     gdc: 10 studies, 569 source columns
retrievers ['qwen', 'magneto']
models     ['ollama__deepseek-v4.1-flash-cloud__tgt-match-retriever']
target     match_retriever
prune      none
retries    3 on an unreadable reply   (tagged into the variant)
repeats    1   runs -> C:\Users\muril\SchemaOracle\results\runs

DRY_RUN = False   *** THIS WILL SEND REQUESTS ***

arms       ['rerank', 'gated', 'paged']
filters    ['none']

3 job(s), all of them priced together in section 2:
  rerank, prune=none           levels=[20]
  gated, prune=none            levels=[20]
  paged, prune=none            levels=[0.0]


## 1. What this will send

`bench.cost` reads the runs on disk and quotes only the columns that are
missing or that a retry could change. Nothing is sent until section 2.


In [18]:
# `levels` is not optional: without it every level on disk is counted, and
# a sweep at w=20, tau=0 was quoted columns belonging to a w=5, tau=0.5 run.
TODO = pd.concat([bench.cost(arm, levels=levels_for(arm)).assign(
    arm=arm, prune=tag or "none") for arm, tag, bench in JOBS],
    ignore_index=True)
display(TODO)
print("")
print(f"this configuration would send "
      f"{int(TODO['columns_todo'].sum()):,} column(s)")


0 run(s) to do
  columns            0
  best case          0 LLM calls
  worst case         0 LLM calls
0 run(s) to do
  columns            0
  best case          0 LLM calls
  worst case         0 LLM calls
0 run(s) to do
  columns            0
  best case          0 LLM calls
  worst case         0 LLM calls


,retriever,model,top_k,runs_todo,runs_partial,columns_todo,columns_stuck,calls_best,calls_worst,arm,prune,threshold
0,qwen,ollama__deepseek-v4.1-flash-cloud__tgt-match-r...,20.0,0,0,0,0,0,0,rerank,none,NaN
1,magneto,ollama__deepseek-v4.1-flash-cloud__tgt-match-r...,20.0,0,0,0,0,0,0,rerank,none,NaN
2,qwen,ollama__deepseek-v4.1-flash-cloud__tgt-match-r...,20.0,0,0,0,0,0,0,gated,none,NaN
3,magneto,ollama__deepseek-v4.1-flash-cloud__tgt-match-r...,20.0,0,0,0,0,0,0,gated,none,NaN
4,qwen,ollama__deepseek-v4.1-flash-cloud__tgt-match-r...,NaN,0,0,0,0,0,0,paged,none,0.0
5,magneto,ollama__deepseek-v4.1-flash-cloud__tgt-match-r...,NaN,0,0,0,0,0,0,paged,none,0.0



this configuration would send 0 column(s)


## 3. Run

In [19]:
# `run` puts the arm and the filter on every row, so the sweeps concatenate
# without anything being added here and no row loses which job made it.
SWEEPS = []
for arm, tag, bench in JOBS:
    print(f"\n=== {label(arm, tag)} " + "=" * 40)
    SWEEPS.append(bench.run(arm, levels=levels_for(arm)))

SWEEP = pd.concat(SWEEPS, ignore_index=True) if SWEEPS else pd.DataFrame()
display(SWEEP)


=== rerank, prune=none ========================================
[qwen 1/10] Cao: 39 x 736 targets (0.0s)
[qwen 2/10] Clark: 16 x 736 targets (0.0s)
[qwen 3/10] Dou: 179 x 736 targets (0.3s)
[qwen 4/10] Gillette: 67 x 736 targets (0.1s)
[qwen 5/10] Huang: 37 x 736 targets (0.0s)
[qwen 6/10] Krug: 54 x 736 targets (0.0s)
[qwen 7/10] McDermott: 22 x 736 targets (0.0s)
[qwen 8/10] Satpathy: 82 x 736 targets (0.1s)
[qwen 9/10] Vasaikar: 45 x 736 targets (0.0s)
[qwen 10/10] Wang: 28 x 736 targets (0.0s)
[magneto 1/10] Cao: 39 x 736 targets (0.0s)
[magneto 2/10] Clark: 16 x 736 targets (0.0s)
[magneto 3/10] Dou: 179 x 736 targets (0.2s)
[magneto 4/10] Gillette: 67 x 736 targets (0.0s)
[magneto 5/10] Huang: 37 x 736 targets (0.0s)
[magneto 6/10] Krug: 54 x 736 targets (0.0s)
[magneto 7/10] McDermott: 22 x 736 targets (0.0s)
[magneto 8/10] Satpathy: 82 x 736 targets (0.1s)
[magneto 9/10] Vasaikar: 45 x 736 targets (0.0s)
[magneto 10/10] Wang: 28 x 736 targets (0.0s)
cached qwen     ollama__dee

,pair,arm,cached,columns,committed,accepted,abstained,errored,complete,calls,tokens_in,tokens_out,wall_s,retriever,model,rep,prune,top_k,threshold
0,Cao,rerank,True,39,39,39,0,0,True,39,38422,6479,0.0,qwen,ollama__deepseek-v4.1-flash-cloud__tgt-match-r...,1,,20.0,NaN
1,Clark,rerank,True,16,16,16,0,0,True,16,16178,2609,0.0,qwen,ollama__deepseek-v4.1-flash-cloud__tgt-match-r...,1,,20.0,NaN
2,Dou,rerank,True,179,179,179,0,0,True,179,174398,28760,0.0,qwen,ollama__deepseek-v4.1-flash-cloud__tgt-match-r...,1,,20.0,NaN
3,Gillette,rerank,True,67,67,67,0,0,True,67,63896,10863,0.0,qwen,ollama__deepseek-v4.1-flash-cloud__tgt-match-r...,1,,20.0,NaN
4,Huang,rerank,True,37,37,37,0,0,True,37,37204,6112,0.0,qwen,ollama__deepseek-v4.1-flash-cloud__tgt-match-r...,1,,20.0,NaN
5,Krug,rerank,True,54,54,54,0,0,True,54,51950,8670,0.0,qwen,ollama__deepseek-v4.1-flash-cloud__tgt-match-r...,1,,20.0,NaN
6,McDermott,rerank,True,22,22,22,0,0,True,22,21640,3568,0.0,qwen,ollama__deepseek-v4.1-flash-cloud__tgt-match-r...,1,,20.0,NaN
7,Satpathy,rerank,True,82,82,82,0,0,True,82,78582,13205,0.0,qwen,ollama__deepseek-v4.1-flash-cloud__tgt-match-r...,1,,20.0,NaN
8,Vasaikar,rerank,True,45,45,45,0,0,True,45,42578,7332,0.0,qwen,ollama__deepseek-v4.1-flash-cloud__tgt-match-r...,1,,20.0,NaN
9,Wang,rerank,True,28,28,28,0,0,True,28,25530,4481,0.0,qwen,ollama__deepseek-v4.1-flash-cloud__tgt-match-r...,1,,20.0,NaN


## 4. Did it work

In [20]:
# Groups by arm and by filter when either varies, so two jobs are never
# pooled into one row.
display(BENCH.health(SWEEP))

no errored columns


runs  columns  calls  committed  abstained  wall_s  calls_per_column
arm    retriever model                                              top_k threshold                                                                      
gated  magneto   ollama__deepseek-v4.1-flash-cloud__tgt-match-re... 20.0  NaN          10      569    569        355        214     0.0              1.00
       qwen      ollama__deepseek-v4.1-flash-cloud__tgt-match-re... 20.0  NaN          10      569    569        482         87     0.0              1.00
paged  magneto   ollama__deepseek-v4.1-flash-cloud__tgt-match-re... NaN   0.0          10      569   6746        414        155     0.0             11.86
       qwen      ollama__deepseek-v4.1-flash-cloud__tgt-match-re... NaN   0.0          10      569   3436        495         74     0.0              6.04
rerank magneto   ollama__deepseek-v4.1-flash-cloud__tgt-match-re... 20.0  NaN          10      569    569        569          0     0.0              1.00
       qwen      ollama__deepseek-v4.1-flash-cloud__tgt-match-re... 20.0  NaN          10      569    569        569          0     0.0              1.00

## 5. Matching quality, and what declining bought

In [21]:
# `score` takes an inventory narrowed to one arm and is told which, so this
# is a loop and not one call over the concatenation. `kind` is the arm; the
# inventory's own `arm` column is the model tag, which is a different thing.
INVS, SCORES = [], []
for arm, tag, bench in JOBS:
    inv = bench.inventory(arm)          # already narrowed to this filter
    if not len(inv):
        print(f"{label(arm, tag)}: nothing on disk")
        continue
    INVS.append(inv)
    SCORES.append(report.score(inv, bench.pairs, results=bench.results,
                               corpus=CORPUS, kind=arm)
                  .assign(kind=arm, prune=tag))

INV = pd.concat(INVS, ignore_index=True) if INVS else pd.DataFrame()
SCORED = pd.concat(SCORES, ignore_index=True) if SCORES else pd.DataFrame()

# Both level columns, because the arms do not share one: `paged` sweeps
# `threshold` and the one-call arms only have `top_k`. Every list is filtered
# by what is actually present, so a single-arm sweep prints one of them.
IDS = ["kind", "prune", "retriever", "model", "top_k", "threshold", "rep"]
QUALITY = IDS + ["n_studies", "n_columns", "n_gold_pairs", "mrr", "recall@1",
                 "recall@20", "top1", "ret_mrr", "ret_recall@20"]
DECIDING = IDS + ["precision_committed", "precision_evaluated", "f1_committed",
                  "n_declined", "n_pruned", "n_model_declined",
                  "n_without_gold", "abstain_precision", "abstain_recall"]
COSTING = IDS + ["calls", "calls_per_column", "calls_median", "calls_max",
                 "calls_share_top", "calls_with_gold", "calls_without_gold",
                 "n_fallback"]

print("ranking quality")
display(SCORED[[c for c in QUALITY if c in SCORED.columns]].round(3))
print("\nwhat declining buys")
display(SCORED[[c for c in DECIDING if c in SCORED.columns]].round(3))
print("\nwhat it costs")
display(SCORED[[c for c in COSTING if c in SCORED.columns]].round(3))

ranking quality


,kind,prune,retriever,model,top_k,threshold,rep,n_studies,n_columns,n_gold_pairs,mrr,recall@1,recall@20,top1,ret_mrr,ret_recall@20
0,rerank,,magneto,anthropic__claude-sonnet-5__tgt-match-retriever,20.0,NaN,1,10,569,259,0.857,0.537,0.772,0.842,0.702,0.772
1,rerank,,magneto,ollama__deepseek-v4-flash-cloud__tgt-match-ret...,20.0,NaN,1,10,499,259,0.524,0.320,0.490,0.503,0.702,0.772
2,rerank,,magneto,ollama__deepseek-v4-pro-cloud__tgt-match-retri...,20.0,NaN,1,10,517,259,0.595,0.363,0.560,0.570,0.702,0.772
3,rerank,,magneto,ollama__deepseek-v4.1-flash-cloud__tgt-match-r...,20.0,NaN,1,10,569,259,0.800,0.486,0.772,0.764,0.702,0.772
4,rerank,,magneto,openrouter__jev-1.13__tgt-match-retriever,20.0,NaN,1,10,569,259,0.857,0.537,0.772,0.842,0.702,0.772
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57,paged,,qwen,ollama__deepseek-v4.1-flash-cloud__tgt-match-r...,NaN,0.5,1,10,569,259,0.906,0.556,0.938,0.873,0.812,0.927
58,paged,,qwen,ollama__gemma4-e4b,NaN,0.0,1,1,16,25,0.786,0.400,0.800,0.714,0.715,0.880
59,paged,,qwen,openrouter__jev-1.13__tgt-match-retriever,NaN,0.0,1,10,569,259,0.917,0.571,0.830,0.897,0.812,0.927
60,paged,,qwen,openrouter__jev-1.13__tgt-match-retriever,NaN,0.5,1,10,569,259,0.883,0.560,0.764,0.879,0.812,0.927



what declining buys


,kind,prune,retriever,model,top_k,threshold,rep,precision_committed,precision_evaluated,f1_committed,n_declined,n_pruned,n_model_declined,n_without_gold,abstain_precision,abstain_recall
0,rerank,,magneto,anthropic__claude-sonnet-5__tgt-match-retriever,20.0,NaN,1,0.244,0.244,0.379,0,0,0,404,NaN,0.000
1,rerank,,magneto,ollama__deepseek-v4-flash-cloud__tgt-match-ret...,20.0,NaN,1,0.166,0.166,0.279,0,0,0,404,NaN,0.000
2,rerank,,magneto,ollama__deepseek-v4-pro-cloud__tgt-match-retri...,20.0,NaN,1,0.182,0.182,0.298,0,0,0,404,NaN,0.000
3,rerank,,magneto,ollama__deepseek-v4.1-flash-cloud__tgt-match-r...,20.0,NaN,1,0.221,0.221,0.343,0,0,0,404,NaN,0.000
4,rerank,,magneto,openrouter__jev-1.13__tgt-match-retriever,20.0,NaN,1,0.244,0.244,0.379,0,0,0,404,NaN,0.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57,paged,,qwen,ollama__deepseek-v4.1-flash-cloud__tgt-match-r...,NaN,0.5,1,0.340,0.253,0.489,145,0,145,404,0.986,0.354
58,paged,,qwen,ollama__gemma4-e4b,NaN,0.0,1,0.625,0.625,0.667,0,0,0,2,NaN,0.000
59,paged,,qwen,openrouter__jev-1.13__tgt-match-retriever,NaN,0.0,1,0.275,0.260,0.420,30,0,30,404,1.000,0.074
60,paged,,qwen,openrouter__jev-1.13__tgt-match-retriever,NaN,0.5,1,0.293,0.257,0.439,74,0,74,404,0.986,0.181



what it costs


,kind,prune,retriever,model,top_k,threshold,rep,calls,calls_per_column,calls_median,calls_max,calls_share_top,calls_with_gold,calls_without_gold,n_fallback
0,rerank,,magneto,anthropic__claude-sonnet-5__tgt-match-retriever,20.0,NaN,1,569,1.00,1.0,1,0.098,1.000,1.000,0
1,rerank,,magneto,ollama__deepseek-v4-flash-cloud__tgt-match-ret...,20.0,NaN,1,499,1.00,1.0,1,0.098,1.000,1.000,75
2,rerank,,magneto,ollama__deepseek-v4-pro-cloud__tgt-match-retri...,20.0,NaN,1,517,1.00,1.0,1,0.099,1.000,1.000,31
3,rerank,,magneto,ollama__deepseek-v4.1-flash-cloud__tgt-match-r...,20.0,NaN,1,569,1.00,1.0,1,0.098,1.000,1.000,9
4,rerank,,magneto,openrouter__jev-1.13__tgt-match-retriever,20.0,NaN,1,569,1.00,1.0,1,0.098,1.000,1.000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57,paged,,qwen,ollama__deepseek-v4.1-flash-cloud__tgt-match-r...,NaN,0.5,1,6074,10.67,1.0,37,0.341,1.448,14.443,0
58,paged,,qwen,ollama__gemma4-e4b,NaN,0.0,1,16,1.00,1.0,1,0.062,1.000,1.000,0
59,paged,,qwen,openrouter__jev-1.13__tgt-match-retriever,NaN,0.0,1,2192,3.85,1.0,37,0.689,1.097,4.978,0
60,paged,,qwen,openrouter__jev-1.13__tgt-match-retriever,NaN,0.5,1,4665,8.20,1.0,37,0.444,1.570,10.906,0


## 6. As a blocking scheme

Only `paged` and `gated` make a block decision, so `rerank` rows are dropped
here rather than shown as zeros.

Pair Completeness is reported twice: `pc_traversed` is the gold the walk was
shown and `pc_accepted` the gold that survived the decision. The gap between
them is recall paid for and then thrown away. Reduction Ratio counts
comparisons saved and cannot see which ones happened, so it rises for free with
every column a filter removes and is not interpretable without precision beside
it.

In [22]:
BLOCKING = IDS + ["pc_traversed", "pc_accepted", "rr", "rr_stop_at_gold"]
CLASSIFIER = IDS + ["block_precision_gold", "block_recall_gold",
                    "tp_gold", "fp_gold", "fn_gold",
                    "n_comparisons", "n_comparisons_brute"]

# Only the arms that make a block decision, rather than the whole frame: a
# rerank row carries these as NaN and printing it would read as a zero.
BLOCKS = (SCORED[SCORED["pc_traversed"].notna()]
          if len(SCORED) and "pc_traversed" in SCORED.columns
          else SCORED.iloc[:0] if len(SCORED) else SCORED)

if len(BLOCKS):
    print("the survey's measures")
    display(BLOCKS[[c for c in BLOCKING if c in BLOCKS.columns]].round(3))
    print("\nthe model as a block classifier, over answerable columns only")
    display(BLOCKS[[c for c in CLASSIFIER if c in BLOCKS.columns]].round(3))
    print("\ngold seen and then discarded, per configuration:")
    lost = (BLOCKS["pc_traversed"] - BLOCKS["pc_accepted"]).round(3)
    for r, l in zip(BLOCKS.to_dict("records"), lost):
        level = r.get("threshold", r.get("top_k"))
        print(f"  {r['kind']:<7} p{r['prune'] or 'none':<12} "
              f"{report.short_model(r['model']):<44} {level}  {l:.3f}")
else:
    print("no arm here makes a block decision, so there is nothing to show.")

the survey's measures


,kind,prune,retriever,model,top_k,threshold,rep,pc_traversed,pc_accepted,rr,rr_stop_at_gold
13,gated,,magneto,anthropic__claude-sonnet-5__tgt-match-retriever,20.0,NaN,1,0.772,0.768,0.973,0.279
14,gated,,magneto,ollama__deepseek-v4-flash-cloud__tgt-match-ret...,20.0,NaN,1,0.598,0.351,0.973,0.222
15,gated,,magneto,ollama__deepseek-v4-pro-cloud__tgt-match-retri...,20.0,NaN,1,0.525,0.506,0.973,0.206
16,gated,,magneto,ollama__deepseek-v4.1-flash-cloud__tgt-match-r...,20.0,NaN,1,0.772,0.772,0.973,0.279
17,gated,,magneto,openrouter__jev-1.13__tgt-match-retriever,20.0,NaN,1,0.772,0.761,0.973,0.279
18,gated,,qwen,anthropic__claude-sonnet-5__tgt-match-retriever,20.0,NaN,1,0.927,0.927,0.973,0.281
19,gated,,qwen,ollama__deepseek-v4-flash-cloud__tgt-match-ret...,20.0,NaN,1,0.927,0.923,0.973,0.281
20,gated,,qwen,ollama__deepseek-v4-pro-cloud__tgt-match-retri...,20.0,NaN,1,0.927,0.927,0.973,0.281
21,gated,,qwen,ollama__deepseek-v4.1-flash-cloud__tgt-match-r...,20.0,NaN,1,0.927,0.927,0.973,0.281
22,gated,,qwen,openrouter__jev-1.13__tgt-match-retriever,20.0,NaN,1,0.927,0.923,0.973,0.281



the model as a block classifier, over answerable columns only


,kind,prune,retriever,model,top_k,threshold,rep,block_precision_gold,block_recall_gold,tp_gold,fp_gold,fn_gold,n_comparisons,n_comparisons_brute
13,gated,,magneto,anthropic__claude-sonnet-5__tgt-match-retriever,20.0,NaN,1,0.917,0.993,144.0,13.0,1.0,11380.0,418784.0
14,gated,,magneto,ollama__deepseek-v4-flash-cloud__tgt-match-ret...,20.0,NaN,1,0.960,0.655,72.0,3.0,38.0,10480.0,385664.0
15,gated,,magneto,ollama__deepseek-v4-pro-cloud__tgt-match-retri...,20.0,NaN,1,0.941,0.950,96.0,6.0,5.0,10280.0,378304.0
16,gated,,magneto,ollama__deepseek-v4.1-flash-cloud__tgt-match-r...,20.0,NaN,1,0.890,1.000,145.0,18.0,0.0,11380.0,418784.0
17,gated,,magneto,openrouter__jev-1.13__tgt-match-retriever,20.0,NaN,1,0.900,0.993,144.0,16.0,1.0,11380.0,418784.0
18,gated,,qwen,anthropic__claude-sonnet-5__tgt-match-retriever,20.0,NaN,1,0.963,1.000,156.0,6.0,0.0,11380.0,418784.0
19,gated,,qwen,ollama__deepseek-v4-flash-cloud__tgt-match-ret...,20.0,NaN,1,0.957,0.994,155.0,7.0,1.0,11380.0,418784.0
20,gated,,qwen,ollama__deepseek-v4-pro-cloud__tgt-match-retri...,20.0,NaN,1,0.957,1.000,156.0,7.0,0.0,11380.0,418784.0
21,gated,,qwen,ollama__deepseek-v4.1-flash-cloud__tgt-match-r...,20.0,NaN,1,0.957,1.000,156.0,7.0,0.0,11380.0,418784.0
22,gated,,qwen,openrouter__jev-1.13__tgt-match-retriever,20.0,NaN,1,0.963,0.994,155.0,6.0,1.0,11380.0,418784.0



gold seen and then discarded, per configuration:
  gated   pnone         claude-sonnet-5 tgt-match-retriever          nan  0.004
  gated   pnone         deepseek-v4-flash-cloud tgt-match-retriever  nan  0.247
  gated   pnone         deepseek-v4-pro-cloud tgt-match-retriever    nan  0.019
  gated   pnone         deepseek-v4.1-flash-cloud tgt-match-retriever nan  0.000
  gated   pnone         jev-1.13 tgt-match-retriever                 nan  0.012
  gated   pnone         claude-sonnet-5 tgt-match-retriever          nan  0.000
  gated   pnone         deepseek-v4-flash-cloud tgt-match-retriever  nan  0.004
  gated   pnone         deepseek-v4-pro-cloud tgt-match-retriever    nan  0.000
  gated   pnone         deepseek-v4.1-flash-cloud tgt-match-retriever nan  0.000
  gated   pnone         jev-1.13 tgt-match-retriever                 nan  0.004
  paged   pnone         claude-sonnet-5 block_cached                 0.0  0.004
  paged   pnone         claude-sonnet-5 tgt-match-retriever         

## 7. Where the calls went

The cost story is a distribution, not a mean. A column that accepts its first
block costs one call and a column with no match anywhere costs
`ceil(n_targets / window)`, so the average sits between two numbers that almost
nothing actually pays.

In [23]:
# Grouped by the job too. Without `kind` a rerank run at one call per column
# would be averaged into a paged run at nineteen, and the mean would describe
# neither.
KEYS = ["retriever", "provider", "arm", "kind", "prune"]
COLUMNS = pd.concat(
    [report.per_column(runs.itertuples(), BENCH.pairs).assign(
        retriever=k[0], model=cache.arm_tag(k[1], k[2]), kind=k[3], prune=k[4])
     for k, runs in INV.groupby(KEYS)],
    ignore_index=True) if len(INV) else pd.DataFrame()

if len(COLUMNS):
    display(COLUMNS.groupby(["kind", "prune", "model"])["api_calls"]
            .describe().round(2))
    print("\ncalls by whether the column had a correct target at all:")
    display(COLUMNS.groupby(["kind", "prune", "model", "has_gold"])["api_calls"]
            .agg(["count", "mean", "median", "max"]).round(2))

count   mean    std  min  25%  50%   75%   max
kind   prune model                                                                                              
gated        anthropic__claude-sonnet-5__tgt-match-retriever     1138.0   1.00   0.00  1.0  1.0  1.0   1.0   1.0
             ollama__deepseek-v4-flash-cloud__tgt-match-retr...  1093.0   1.00   0.00  1.0  1.0  1.0   1.0   1.0
             ollama__deepseek-v4-pro-cloud__tgt-match-retriever  1083.0   1.00   0.00  1.0  1.0  1.0   1.0   1.0
             ollama__deepseek-v4.1-flash-cloud__tgt-match-re...  1138.0   1.00   0.00  1.0  1.0  1.0   1.0   1.0
             openrouter__jev-1.13__tgt-match-retriever           1138.0   1.00   0.00  1.0  1.0  1.0   1.0   1.0
paged        anthropic__claude-sonnet-5                           569.0   6.90  12.86  1.0  1.0  1.0   2.0  37.0
             anthropic__claude-sonnet-5__block_cached            1138.0   7.32  12.85  1.0  1.0  1.0   3.0  37.0
             anthropic__claude-sonnet-5__block_cached__tgt-m...   569.0   3.42  10.88  1.0  1.0  1.0   1.0  74.0
             anthropic__claude-sonnet-5__tgt-match-retriever     1138.0   7.26  15.43  1.0  1.0  1.0   3.0  74.0
             anthropic__claude-sonnet-5__tgt-match-retriever...    39.0   1.28   1.02  1.0  1.0  1.0   1.0   6.0
             ollama__deepseek-v4-flash-cloud__tgt-match-retr...  1076.0  23.88  28.12  1.0  1.0  4.0  37.0  74.0
             ollama__deepseek-v4-pro-cloud__tgt-match-retriever  1090.0   3.25   7.44  1.0  1.0  1.0   1.0  37.0
             ollama__deepseek-v4.1-flash-cloud__oneshot-none...   569.0   3.23   8.08  1.0  1.0  1.0   1.0  37.0
             ollama__deepseek-v4.1-flash-cloud__tgt-match-re...  1138.0   8.95  14.42  1.0  1.0  1.0   5.0  37.0
             ollama__gemma4-e4b                                    16.0   1.00   0.00  1.0  1.0  1.0   1.0   1.0
             openrouter__jev-1.13__tgt-match-retriever           1138.0   3.65   7.78  1.0  1.0  1.0   2.0  37.0
rerank       anthropic__claude-sonnet-5                          1138.0   1.00   0.00  1.0  1.0  1.0   1.0   1.0
             anthropic__claude-sonnet-5__tgt-match-retriever     1138.0   1.00   0.00  1.0  1.0  1.0   1.0   1.0
             ollama__deepseek-v4-flash-cloud__tgt-match-retr...  1068.0   1.00   0.00  1.0  1.0  1.0   1.0   1.0
             ollama__deepseek-v4-pro-cloud__tgt-match-retriever  1086.0   1.00   0.00  1.0  1.0  1.0   1.0   1.0
             ollama__deepseek-v4.1-flash-cloud__tgt-match-re...  1138.0   1.00   0.00  1.0  1.0  1.0   1.0   1.0
             openrouter__jev-1.13__tgt-match-retriever           1138.0   1.00   0.00  1.0  1.0  1.0   1.0   1.0


calls by whether the column had a correct target at all:


count   mean  median  max
kind   prune model                                              has_gold                           
gated        anthropic__claude-sonnet-5__tgt-match-retriever    False       808   1.00     1.0    1
                                                                True        330   1.00     1.0    1
             ollama__deepseek-v4-flash-cloud__tgt-match-retr... False       808   1.00     1.0    1
                                                                True        285   1.00     1.0    1
             ollama__deepseek-v4-pro-cloud__tgt-match-retriever False       808   1.00     1.0    1
                                                                True        275   1.00     1.0    1
             ollama__deepseek-v4.1-flash-cloud__tgt-match-re... False       808   1.00     1.0    1
                                                                True        330   1.00     1.0    1
             openrouter__jev-1.13__tgt-match-retriever          False       808   1.00     1.0    1
                                                                True        330   1.00     1.0    1
paged        anthropic__claude-sonnet-5                         False       404   9.27     1.0   37
                                                                True        165   1.08     1.0   10
             anthropic__claude-sonnet-5__block_cached           False       808   9.89     1.0   37
                                                                True        330   1.03     1.0    3
             anthropic__claude-sonnet-5__block_cached__tgt-m... False       404   4.40     1.0   74
                                                                True        165   1.04     1.0    3
             anthropic__claude-sonnet-5__tgt-match-retriever    False       808   9.79     1.0   74
                                                                True        330   1.06     1.0    5
             anthropic__claude-sonnet-5__tgt-match-retriever... False        10   2.10     1.0    6
                                                                True         29   1.00     1.0    1
             ollama__deepseek-v4-flash-cloud__tgt-match-retr... False       808  31.00    37.0   74
                                                                True        268   2.43     1.0   74
             ollama__deepseek-v4-pro-cloud__tgt-match-retriever False       808   4.02     1.0   37
                                                                True        282   1.04     1.0    8
             ollama__deepseek-v4.1-flash-cloud__oneshot-none... False       404   4.13     1.0   37
                                                                True        165   1.01     1.0    2
             ollama__deepseek-v4.1-flash-cloud__tgt-match-re... False       808  12.05     1.0   37
                                                                True        330   1.34     1.0   37
             ollama__gemma4-e4b                                 False         2   1.00     1.0    1
                                                                True         14   1.00     1.0    1
             openrouter__jev-1.13__tgt-match-retriever          False       808   4.69     1.0   37
                                                                True        330   1.09     1.0   13
rerank       anthropic__claude-sonnet-5                         False       808   1.00     1.0    1
                                                                True        330   1.00     1.0    1
             anthropic__claude-sonnet-5__tgt-match-retriever    False       808   1.00     1.0    1
                                                                True        330   1.00     1.0    1
             ollama__deepseek-v4-flash-cloud__tgt-match-retr... False       808   1.00     1.0    1
                                                                True        260   1.00     1.0    1
             ollama__deepseek-v4-pro-cloud__tgt-match-retriever False     

## 8. Spread across repeats

A gap between two models smaller than either one's own spread is not a gap, and
one run cannot say which case it is in.

In [24]:
if len(SCORED) and SCORED["rep"].nunique() > 1:
    display(report.across_repeats(SCORED).round(4))
else:
    print(f"{SCORED['rep'].nunique() if len(SCORED) else 0} repeat(s) on disk; "
          "set REPS above 1 to measure the noise floor.")

,retriever,provider,arm,model,kind,prune,top_k,window,threshold,mrr_mean,mrr_std,reps,recall@1_mean,recall@1_std,recall@1_count,top1_mean,top1_std,top1_count,precision_committed_mean,precision_committed_std,precision_committed_count,precision_evaluated_mean,precision_evaluated_std,precision_evaluated_count,mrr_abstain_mean,mrr_abstain_std,mrr_abstain_count,calls_mean,calls_std,calls_count
0,magneto,anthropic,claude-sonnet-5__block_cached,anthropic__claude-sonnet-5__block_cached,paged,,NaN,20.0,0.0,0.8657,NaN,1,0.5405,NaN,1,0.8485,NaN,1,0.2857,NaN,1,0.2460,NaN,1,0.3899,NaN,1,4224.0,NaN,1
1,magneto,anthropic,claude-sonnet-5__tgt-match-retriever,anthropic__claude-sonnet-5__tgt-match-retriever,gated,,20.0,NaN,NaN,0.8477,NaN,1,0.5290,NaN,1,0.8303,NaN,1,0.4041,NaN,1,0.2408,NaN,1,0.6360,NaN,1,569.0,NaN,1
2,magneto,anthropic,claude-sonnet-5__tgt-match-retriever,anthropic__claude-sonnet-5__tgt-match-retriever,paged,,NaN,20.0,0.0,0.8914,NaN,1,0.5598,NaN,1,0.8788,NaN,1,0.2929,NaN,1,0.2548,NaN,1,0.3885,NaN,1,4495.0,NaN,1
3,magneto,anthropic,claude-sonnet-5__tgt-match-retriever,anthropic__claude-sonnet-5__tgt-match-retriever,rerank,,20.0,NaN,NaN,0.8571,NaN,1,0.5367,NaN,1,0.8424,NaN,1,0.2443,NaN,1,0.2443,NaN,1,0.2485,NaN,1,569.0,NaN,1
4,magneto,ollama,deepseek-v4-flash-cloud__tgt-match-retriever,ollama__deepseek-v4-flash-cloud__tgt-match-ret...,gated,,20.0,NaN,NaN,0.4084,NaN,1,0.2510,NaN,1,0.3939,NaN,1,0.4114,NaN,1,0.1851,NaN,1,0.7298,NaN,1,524.0,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56,qwen,openrouter,jev-1.13__tgt-match-retriever,openrouter__jev-1.13__tgt-match-retriever,gated,,20.0,NaN,NaN,0.9000,NaN,1,0.5598,NaN,1,0.8788,NaN,1,0.3349,NaN,1,0.2548,NaN,1,0.4930,NaN,1,569.0,NaN,1
57,qwen,openrouter,jev-1.13__tgt-match-retriever,openrouter__jev-1.13__tgt-match-retriever,paged,,NaN,20.0,0.0,0.9172,NaN,1,0.5714,NaN,1,0.8970,NaN,1,0.2746,NaN,1,0.2601,NaN,1,0.3187,NaN,1,2192.0,NaN,1
58,qwen,openrouter,jev-1.13__tgt-match-retriever,openrouter__jev-1.13__tgt-match-retriever,paged,,NaN,20.0,0.5,0.8828,NaN,1,0.5598,NaN,1,0.8788,NaN,1,0.2929,NaN,1,0.2566,NaN,1,0.3843,NaN,1,4665.0,NaN,1
59,qwen,openrouter,jev-1.13__tgt-match-retriever,openrouter__jev-1.13__tgt-match-retriever,paged,,NaN,20.0,0.7,0.7535,NaN,1,0.4788,NaN,1,0.7515,NaN,1,0.3669,NaN,1,0.2373,NaN,1,0.5964,NaN,1,9993.0,NaN,1
